# FULL Benchmark Accuracy Evaluation — Enterprise BFSI Client MDM

**Purpose:** Evaluate the existing FULL Azure Databricks outputs against the synthetic benchmark's hidden `entity_key` ground truth. This notebook is read-only against operational tables: it does not rerun or modify matching, stewardship, Gold, or source data.

**Ground-truth isolation:** The hidden `entity_key` is used only in this evaluation notebook. Never add it to Silver, blocking features, `mdm.match_results`, Golden Records, or serving tables.

## Before running

1. Upload the frozen FULL `ground_truth.csv` to a restricted evaluation-only ADLS Gen2 folder or governed Unity Catalog Volume.
2. Run the configuration cell and set the `GROUND_TRUTH_PATH` widget to that file's URI.
3. Verify that the four operational tables below represent the same completed FULL run.
4. Run all cells. The notebook stops if row counts or record mappings do not reconcile.

## Metrics

- Blocking recall.
- Automatic pairwise MATCH precision, end-to-end recall and F1.
- Final Golden Record cluster pairwise precision, recall and F1.
- Over-merged master cluster and split-entity counts.

All metrics are pairwise identity-resolution measures over trusted Silver. They do not measure factual correctness of survivorship attributes such as phone, email, address, or AUM.

## Latest recorded FULL result snapshot

The following are the results from the executed evaluation notebook uploaded on 2026-10-13. Re-run this notebook to refresh them for a different checkpoint.

| Metric | Recorded FULL result |
|---|---:|
| Blocking recall | 99.4769% |
| Automatic-match precision | 99.6764% |
| End-to-end pairwise recall | 89.3703% |
| Automatic-match F1 | 94.2424% |
| Final cluster pairwise precision | 99.6424% |
| Final cluster pairwise recall | 93.7916% |
| Final cluster pairwise F1 | 96.6285% |
| Over-merged master clusters | 1,218 |
| Underlying entities split across masters | 82,484 |

The recorded result's detailed row counts are printed by the evaluation cells below. Do not interpret candidate decision shares as accuracy, and do not substitute DEV benchmark metrics for FULL results.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# CONFIGURATION — existing FULL run tables are read-only
# ============================================================
CATALOG = "bfsi_mdm_dev"

SILVER_TABLE = f"{CATALOG}.silver.all_clients"
CANDIDATE_TABLE = f"{CATALOG}.mdm.candidate_pairs"
MATCH_RESULTS_TABLE = f"{CATALOG}.mdm.match_results"
SOURCE_MAP_TABLE = f"{CATALOG}.gold.client_source_map"

# Provide the hidden ground truth only to this evaluation notebook.
# The path is supplied by widget to avoid committing account/container names.
dbutils.widgets.text(
    "GROUND_TRUTH_PATH",
    "",
    "FULL ground_truth.csv URI (restricted ADLS Gen2 path or Unity Catalog Volume)",
)
dbutils.widgets.text(
    "RESULT_OUTPUT_PATH",
    "",
    "Optional evaluation-only output folder URI (leave blank to only display)",
)

GROUND_TRUTH_PATH = dbutils.widgets.get("GROUND_TRUTH_PATH").strip()
RESULT_OUTPUT_PATH = dbutils.widgets.get("RESULT_OUTPUT_PATH").strip()

if not GROUND_TRUTH_PATH:
    raise ValueError(
        "Set the GROUND_TRUTH_PATH widget to the exact URI of the FULL "
        "ground_truth.csv before running all cells."
    )

print("Catalog:", CATALOG)
print("Silver:", SILVER_TABLE)
print("Candidate pairs:", CANDIDATE_TABLE)
print("Match results:", MATCH_RESULTS_TABLE)
print("Gold source map:", SOURCE_MAP_TABLE)
print("Ground-truth path configured: yes")
print("Output:", "display only" if not RESULT_OUTPUT_PATH else "aggregate report export enabled")


## 1. Load and validate ground truth and trusted Silver

The evaluator joins records using `(source_system, source_id)`. Ground-truth identifiers remain in this evaluation-only DataFrame.

In [ ]:
gt_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(GROUND_TRUTH_PATH)
)

required_gt_columns = {"source_system", "source_id", "entity_key"}
missing_columns = required_gt_columns - set(gt_raw.columns)
if missing_columns:
    raise ValueError(f"Ground-truth file is missing columns: {sorted(missing_columns)}")

gt = gt_raw.select(
    F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
    F.trim(F.col("source_id").cast("string")).alias("source_id"),
    F.trim(F.col("entity_key").cast("string")).alias("entity_key"),
)

gt_rows = gt.count()
invalid_gt_rows = gt.filter(
    F.col("source_system").isNull() | (F.col("source_system") == "")
    | F.col("source_id").isNull() | (F.col("source_id") == "")
    | F.col("entity_key").isNull() | (F.col("entity_key") == "")
).count()
if invalid_gt_rows:
    raise ValueError(f"Ground truth contains {invalid_gt_rows:,} blank/null mappings.")

duplicate_gt_keys = (
    gt.groupBy("source_system", "source_id")
      .count()
      .filter(F.col("count") > 1)
      .limit(1)
      .count()
)
if duplicate_gt_keys:
    raise ValueError("Ground truth has duplicate (source_system, source_id) mappings.")

silver_keys = (
    spark.table(SILVER_TABLE)
    .select(
        F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
        F.trim(F.col("source_id").cast("string")).alias("source_id"),
    )
    .withColumn("record_key", F.concat_ws("::", "source_system", "source_id"))
)

silver_rows = silver_keys.count()
duplicate_silver_keys = (
    silver_keys.groupBy("source_system", "source_id")
               .count()
               .filter(F.col("count") > 1)
               .limit(1)
               .count()
)
if duplicate_silver_keys:
    raise ValueError("Silver has duplicate source keys; evaluation mapping would be ambiguous.")

silver_without_truth = (
    silver_keys.select("source_system", "source_id")
    .join(gt.select("source_system", "source_id"),
          ["source_system", "source_id"], "left_anti")
    .count()
)
truth_outside_silver = (
    gt.select("source_system", "source_id")
      .join(silver_keys.select("source_system", "source_id"),
            ["source_system", "source_id"], "left_anti")
      .count()
)
if silver_without_truth:
    raise ValueError(
        f"{silver_without_truth:,} Silver records are missing ground truth. "
        "Check the FULL benchmark version before evaluating."
    )

truth_silver = (
    silver_keys.join(gt, ["source_system", "source_id"], "inner")
    .select("source_system", "source_id", "record_key", "entity_key")
)
eligible_rows = truth_silver.count()
eligible_entities = truth_silver.select("entity_key").distinct().count()
if eligible_rows != silver_rows:
    raise ValueError(f"Coverage mismatch: eligible={eligible_rows:,}, Silver={silver_rows:,}.")

print(f"Ground-truth mapping rows loaded       : {gt_rows:,}")
print(f"Invalid/blank ground-truth rows         : {invalid_gt_rows:,}")
print(f"Trusted Silver records                 : {silver_rows:,}")
print(f"Silver records missing ground truth    : {silver_without_truth:,}")
print(f"Ground-truth records outside Silver    : {truth_outside_silver:,}")
print(f"Eligible records for accuracy scoring  : {eligible_rows:,}")
print(f"Underlying entities represented in Silver: {eligible_entities:,}")


## 2. Calculate true-pair denominator without materializing every pair

For an entity with `n` eligible source records, there are `n × (n-1) / 2` true record pairs. Summing by hidden entity gives the denominator without generating all true pairs.

In [ ]:
entity_sizes = truth_silver.groupBy("entity_key").agg(F.count(F.lit(1)).alias("n"))
choose_two = (
    (F.col("n").cast("long") * (F.col("n").cast("long") - F.lit(1))) / F.lit(2)
).cast("long")
true_pairs_total = int(
    entity_sizes.agg(F.sum(choose_two).alias("true_pairs_total")).first()["true_pairs_total"] or 0
)
if true_pairs_total <= 0:
    raise ValueError("No true record pairs found. Check ground truth and Silver population.")
print(f"Total true record pairs among trusted Silver: {true_pairs_total:,}")

record_truth = truth_silver.select("record_key", "entity_key")
truth_a = record_truth.select(
    F.col("record_key").alias("record_key_a"),
    F.col("entity_key").alias("entity_key_a"),
)
truth_b = record_truth.select(
    F.col("record_key").alias("record_key_b"),
    F.col("entity_key").alias("entity_key_b"),
)


## 3. Evaluate candidate blocking recall

A true pair is captured by blocking when both candidate endpoints have the same hidden `entity_key`.

In [ ]:
candidate_pairs = spark.table(CANDIDATE_TABLE).select(
    F.col("record_key_a").cast("string").alias("record_key_a"),
    F.col("record_key_b").cast("string").alias("record_key_b"),
)

candidate_labeled = (
    candidate_pairs.join(truth_a, "record_key_a", "left")
                   .join(truth_b, "record_key_b", "left")
)
candidate_agg = candidate_labeled.agg(
    F.count(F.lit(1)).alias("candidate_rows"),
    F.sum(
        F.when(F.col("entity_key_a").isNull() | F.col("entity_key_b").isNull(), 1).otherwise(0)
    ).alias("candidate_rows_missing_truth"),
    F.sum(
        F.when(F.col("entity_key_a") == F.col("entity_key_b"), 1).otherwise(0)
    ).alias("true_candidate_pairs"),
).first()

candidate_rows = int(candidate_agg["candidate_rows"] or 0)
candidate_missing_truth = int(candidate_agg["candidate_rows_missing_truth"] or 0)
true_candidate_pairs = int(candidate_agg["true_candidate_pairs"] or 0)
if candidate_missing_truth:
    raise ValueError(f"{candidate_missing_truth:,} candidate rows have endpoints absent from the truth mapping.")

missed_by_blocking = true_pairs_total - true_candidate_pairs
if missed_by_blocking < 0:
    raise ValueError("True candidates exceed the unique true-pair denominator; check duplicate candidate edges.")
blocking_recall = true_candidate_pairs / true_pairs_total if true_pairs_total else 0.0

print(f"Candidate pair rows                    : {candidate_rows:,}")
print(f"True pairs present in candidates       : {true_candidate_pairs:,}")
print(f"True pairs missed by blocking          : {missed_by_blocking:,}")
print(f"FULL blocking recall                   : {blocking_recall * 100:.4f}%")


## 4. Evaluate saved direct matching decisions

Automatic `MATCH` is treated as the accepted pairwise decision. `REVIEW` is unresolved, so true REVIEW pairs are reported separately and do not count as automatic true positives. Recall's denominator includes all true pairs in trusted Silver, including pairs missed during blocking.

In [ ]:
match_results = spark.table(MATCH_RESULTS_TABLE).select(
    F.col("record_key_a").cast("string").alias("record_key_a"),
    F.col("record_key_b").cast("string").alias("record_key_b"),
    F.upper(F.trim(F.col("decision").cast("string"))).alias("decision"),
)
match_labeled = (
    match_results.join(truth_a, "record_key_a", "left")
                 .join(truth_b, "record_key_b", "left")
)
match_agg = match_labeled.agg(
    F.count(F.lit(1)).alias("match_rows"),
    F.sum(
        F.when(F.col("entity_key_a").isNull() | F.col("entity_key_b").isNull(), 1).otherwise(0)
    ).alias("match_rows_missing_truth"),
    F.sum(
        F.when((F.col("decision") == "MATCH") & (F.col("entity_key_a") == F.col("entity_key_b")), 1).otherwise(0)
    ).alias("tp"),
    F.sum(
        F.when((F.col("decision") == "MATCH") & (F.col("entity_key_a") != F.col("entity_key_b")), 1).otherwise(0)
    ).alias("fp"),
    F.sum(
        F.when((F.col("decision") == "REVIEW") & (F.col("entity_key_a") == F.col("entity_key_b")), 1).otherwise(0)
    ).alias("true_review_pairs"),
    F.sum(
        F.when((F.col("decision") == "NO_MATCH") & (F.col("entity_key_a") == F.col("entity_key_b")), 1).otherwise(0)
    ).alias("true_no_match_pairs"),
).first()

match_rows = int(match_agg["match_rows"] or 0)
match_missing_truth = int(match_agg["match_rows_missing_truth"] or 0)
tp = int(match_agg["tp"] or 0)
fp = int(match_agg["fp"] or 0)
true_review_pairs = int(match_agg["true_review_pairs"] or 0)
true_no_match_pairs = int(match_agg["true_no_match_pairs"] or 0)
if match_missing_truth:
    raise ValueError(f"{match_missing_truth:,} matching result rows have endpoints absent from truth.")
if match_rows != candidate_rows:
    raise ValueError(f"Candidate/result row-count mismatch: candidates={candidate_rows:,}, results={match_rows:,}.")

fn = true_pairs_total - tp
true_candidates_not_auto_matched = true_candidate_pairs - tp
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / true_pairs_total if true_pairs_total else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print("PAIRWISE AUTOMATIC-MATCH METRICS")
print("--------------------------------")
print(f"True positives (MATCH and same entity) : {tp:,}")
print(f"False positives (MATCH, different entity): {fp:,}")
print(f"False negatives (true pairs not MATCH) : {fn:,}")
print(f"  - True pairs missed by blocking       : {missed_by_blocking:,}")
print(f"  - True candidate pairs not auto-MATCH : {true_candidates_not_auto_matched:,}")
print(f"True pairs marked REVIEW               : {true_review_pairs:,}")
print(f"True pairs marked NO_MATCH             : {true_no_match_pairs:,}")
print(f"Precision                              : {precision * 100:.4f}%")
print(f"Recall (end-to-end pairwise)            : {recall * 100:.4f}%")
print(f"F1-score                               : {f1 * 100:.4f}%")


## 5. Evaluate final Golden Record clusters

A predicted pair shares a master if both records have the same `master_client_id`. Cluster pairwise metrics are computed from group counts rather than enumerating all record pairs.

In [ ]:
source_map = (
    spark.table(SOURCE_MAP_TABLE)
    .select(
        F.trim(F.col("master_client_id").cast("string")).alias("master_client_id"),
        F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
        F.trim(F.col("source_id").cast("string")).alias("source_id"),
    )
)
source_map_rows = source_map.count()
source_map_duplicate_keys = (
    source_map.groupBy("source_system", "source_id")
              .count()
              .filter(F.col("count") > 1)
              .limit(1)
              .count()
)
if source_map_duplicate_keys:
    raise ValueError("Gold source map assigns a source record more than once.")

cluster_truth = source_map.join(
    truth_silver.select("source_system", "source_id", "entity_key"),
    ["source_system", "source_id"],
    "left",
)
cluster_unmapped_rows = cluster_truth.filter(
    F.col("master_client_id").isNull() | F.col("entity_key").isNull()
).count()
if cluster_unmapped_rows:
    raise ValueError(f"{cluster_unmapped_rows:,} source-map rows lack a master ID or ground truth.")
if source_map_rows != eligible_rows:
    raise ValueError(
        f"Gold source-map rows ({source_map_rows:,}) do not equal trusted Silver "
        f"rows ({eligible_rows:,}); do not interpret cluster metrics."
    )

master_sizes = source_map.groupBy("master_client_id").agg(F.count(F.lit(1)).alias("n"))
predicted_cluster_pairs = int(
    master_sizes.agg(F.sum(choose_two).alias("pairs")).first()["pairs"] or 0
)
master_entity_sizes = cluster_truth.groupBy("master_client_id", "entity_key").agg(
    F.count(F.lit(1)).alias("n")
)
cluster_tp = int(
    master_entity_sizes.agg(F.sum(choose_two).alias("pairs")).first()["pairs"] or 0
)
cluster_fp = predicted_cluster_pairs - cluster_tp
cluster_fn = true_pairs_total - cluster_tp
cluster_precision = cluster_tp / predicted_cluster_pairs if predicted_cluster_pairs else 0.0
cluster_recall = cluster_tp / true_pairs_total if true_pairs_total else 0.0
cluster_f1 = (
    2 * cluster_precision * cluster_recall / (cluster_precision + cluster_recall)
    if (cluster_precision + cluster_recall) else 0.0
)

overmerged_master_clusters = (
    cluster_truth.groupBy("master_client_id")
                 .agg(F.countDistinct("entity_key").alias("true_entity_count"))
                 .filter(F.col("true_entity_count") > 1)
                 .count()
)
split_true_entities = (
    cluster_truth.groupBy("entity_key")
                 .agg(F.countDistinct("master_client_id").alias("master_count"))
                 .filter(F.col("master_count") > 1)
                 .count()
)
master_id_count = source_map.select("master_client_id").distinct().count()

print("FINAL CLUSTER METRICS")
print("---------------------")
print(f"Master IDs in source map                 : {master_id_count:,}")
print(f"Predicted same-master record pairs       : {predicted_cluster_pairs:,}")
print(f"Correct same-entity, same-master pairs   : {cluster_tp:,}")
print(f"Incorrect same-master pairs (over-merge) : {cluster_fp:,}")
print(f"Same-entity pairs split across masters   : {cluster_fn:,}")
print(f"Cluster pairwise precision              : {cluster_precision * 100:.4f}%")
print(f"Cluster pairwise recall                 : {cluster_recall * 100:.4f}%")
print(f"Cluster pairwise F1                      : {cluster_f1 * 100:.4f}%")
print(f"Master clusters containing >1 true entity: {overmerged_master_clusters:,}")
print(f"True entities split across >1 master     : {split_true_entities:,}")


## 6. Compact scorecard and optional aggregate export

The output contains aggregate metrics only—no record-level identifiers or hidden entity keys. If you export the scorecard, use a restricted evaluation-only folder.

In [ ]:
def pct(value):
    return f"{value * 100:.4f}%"

scorecard_rows = [
    ("Population", "Ground-truth mapping rows", f"{gt_rows:,}"),
    ("Population", "Trusted Silver records", f"{silver_rows:,}"),
    ("Population", "Ground-truth rows outside Silver", f"{truth_outside_silver:,}"),
    ("Population", "Entities represented in Silver", f"{eligible_entities:,}"),
    ("Population", "True record pairs", f"{true_pairs_total:,}"),
    ("Blocking", "Candidate pairs", f"{candidate_rows:,}"),
    ("Blocking", "True candidate pairs", f"{true_candidate_pairs:,}"),
    ("Blocking", "True pairs missed", f"{missed_by_blocking:,}"),
    ("Blocking", "Blocking recall", pct(blocking_recall)),
    ("Direct matching", "True positives", f"{tp:,}"),
    ("Direct matching", "False positives", f"{fp:,}"),
    ("Direct matching", "False negatives", f"{fn:,}"),
    ("Direct matching", "True pairs marked REVIEW", f"{true_review_pairs:,}"),
    ("Direct matching", "True pairs marked NO_MATCH", f"{true_no_match_pairs:,}"),
    ("Direct matching", "Precision", pct(precision)),
    ("Direct matching", "End-to-end recall", pct(recall)),
    ("Direct matching", "F1", pct(f1)),
    ("Final clusters", "Master IDs", f"{master_id_count:,}"),
    ("Final clusters", "Over-merged clusters", f"{overmerged_master_clusters:,}"),
    ("Final clusters", "Split underlying entities", f"{split_true_entities:,}"),
    ("Final clusters", "Pairwise precision", pct(cluster_precision)),
    ("Final clusters", "Pairwise recall", pct(cluster_recall)),
    ("Final clusters", "Pairwise F1", pct(cluster_f1)),
]
scorecard = spark.createDataFrame(
    scorecard_rows,
    schema=T.StructType([
        T.StructField("section", T.StringType(), False),
        T.StructField("metric", T.StringType(), False),
        T.StructField("value", T.StringType(), False),
    ]),
)
display(scorecard)

if RESULT_OUTPUT_PATH:
    (
        scorecard.coalesce(1)
        .write.mode("overwrite")
        .option("header", "true")
        .csv(RESULT_OUTPUT_PATH)
    )
    print("Aggregate scorecard written to the configured evaluation-only folder.")
else:
    print("No output path supplied; scorecard displayed only.")


## 7. Interpretation and reporting safeguards

- Do not call the candidate decision distribution an accuracy metric.
- Do not use DEV metrics as FULL cloud metrics.
- Keep direct pairwise metrics separate from final cluster metrics.
- `entity_key` is evaluation-only; it must never enter operational matching or serving.
- This evaluates synthetic identity resolution, not factual attribute correctness of Golden Records.
- Review cases are unresolved for automatic matching. If evaluating after additional stewardship, label that as a separate checkpoint.
- Stop if any ground-truth coverage, candidate/result count, benchmark version, or source-map reconciliation fails.
